# MCMC GPU TEST SCRIPT 

Author: Tyler Le 

Date: 05/02/2026

Comparing CPU and GPU methods to see if they return the same or the same (within tolerance) results

The difference should be the runtime and execution time

In [1]:
import cupy as cp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.preprocessing import QuantileTransformer
from MCMC_GPU.QuantileTransformer_gpu import NormalScoreTransformation


In [2]:
df = pd.read_csv('./data/BindSchadler_Macayeal_IceStreams.csv')
bed = np.load('./data/bindshadler_macayeal/LargeScaleChain/125068/bed_29910k.npy')


In [3]:

x_uniq = np.unique(df.x)
y_uniq = np.unique(df.y)

xmin = np.min(x_uniq)
xmax = np.max(x_uniq)
ymin = np.min(y_uniq)
ymax = np.max(y_uniq)

cols = len(x_uniq)
rows = len(y_uniq)
seed = 0
resolution = 500

xx, yy = np.meshgrid(x_uniq, y_uniq)

dhdt = df['dhdt'].values.reshape(xx.shape)
smb = df['smb'].values.reshape(xx.shape)
velx = df['velx'].values.reshape(xx.shape)
vely = df['vely'].values.reshape(xx.shape)
bedmap_mask = df['bedmap_mask'].values.reshape(xx.shape)
bedmachine_thickness = df['bedmachine_thickness'].values.reshape(xx.shape)
bedmap_surf = df['bedmap_surf'].values.reshape(xx.shape)
highvel_mask = df['highvel_mask'].values.reshape(xx.shape)
bedmap_bed = df['bedmap_bed'].values.reshape(xx.shape)
cond_bed = np.where(bedmap_mask == 1, df['bed'].values.reshape(xx.shape), bedmap_bed)
df['cond_bed'] = cond_bed.flatten()
data_mask = ~np.isnan(cond_bed)


# Topography Mass Conservation Mass Residual in Numpy vs. Cupy

MC_res.py is written in C to compute gradients manually. 

In [4]:
from MCMC_GPU import MC_res
from gstatsMCMC import Topography

In [7]:
base_mcr = Topography.get_mass_conservation_residual(bed, bedmap_surf, velx, vely, dhdt, smb, resolution)
cupy_mcr = MC_res.get_mass_conservation_residual_fused(bed, bedmap_surf, velx, vely, dhdt, smb, resolution)

In [8]:
cond_base_mcr = Topography.get_mass_conservation_residual(cond_bed, bedmap_surf, velx, vely, dhdt, smb, resolution)
cond_cupy_mcr = MC_res.get_mass_conservation_residual_fused(cond_bed, bedmap_surf, velx, vely, dhdt, smb, resolution)

Let's get the Mass Conservation Loss Residual

In [9]:
mcr_difference = np.nansum(cp.asnumpy(cupy_mcr) - base_mcr)
print(f"Difference between two Mass Conservation Loss methods {mcr_difference} on a LSC bed")

cond_mcr_difference = np.nansum(cp.asnumpy(cond_cupy_mcr) - cond_base_mcr)
print(f"Difference between two Mass Conservation Loss methods {cond_mcr_difference} on a conditioned bed")

Difference between two Mass Conservation Loss methods 5.910938405406796e-12 on a LSC bed
Difference between two Mass Conservation Loss methods -1.8329782136561334e-13 on a conditioned bed


In [10]:
del cupy_mcr
del cond_cupy_mcr

In [11]:
n_trials = 1000
start_time = time.perf_counter()

# Code to measure
for i in range(n_trials):
    cond_base_mcr = Topography.get_mass_conservation_residual(cond_bed, bedmap_surf, velx, vely, dhdt, smb, resolution)

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"CPU Execution time: {execution_time:.6f} seconds")



start_time = time.perf_counter()

# Code to measure
for i in range(n_trials):
    cond_cupy_mcr = MC_res.get_mass_conservation_residual_fused(cond_bed, bedmap_surf, velx, vely, dhdt, smb, resolution)

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"GPU Execution time: {execution_time:.6f} seconds")



CPU Execution time: 5.400441 seconds
GPU Execution time: 2.372784 seconds


The difference is extremely small, $10^{-13}$. This small difference makes sense because the arrays are stored in different memory address, in which their values are not always absolute the same. 

# QT CuPy vs sklearn

## QuantileTransformer transforms the features to follow a *uniform or a normal* distribution. 

    
[scikit-learn QuantileTransformer Source](https://github.com/scikit-learn/scikit-learn/blob/fe2edb3cdbd75ae4e662fda67dcb19277258792b/sklearn/preprocessing/_data.py#L2650)


quantiles_ : ndarray of shape (n_quantiles, n_features)
        The values corresponding the quantiles of reference.

references_ : ndarray of shape (n_quantiles, )
        Quantiles of references.
        
For example:

quantiles_  = [0.3,  1.2,  4.7,  11.0,  38.0, ...]   # original data values 

references_ = [0.0,  0.1,  0.2,   0.3,   0.4, ...]   # Quantile positions


## In T3 & T4, we use QT to normalize our conditioning data, saved to 'Nbed'.

For example, on conditioned beds are transformed/normalized:
``` 
    bed_tosim = nst_trans.transform(bed_c.reshape(-1,1)).reshape(self.xx.shape)
```

This normalization is then perturbed in T3, or perform SGS on normalized conditioned bed in T4. 
```
    newsim = sgs(self.xx, self.yy, bed_tosim, vario, rad, neighbors, sim_mask = sim_mask, seed=rng)
 
```

Once changed, and performed MCR, we inverse transform back to the original cond_bed
```
    bed_next = nst_trans.inverse_transform(newsim.reshape(-1,1)).reshape(rows,cols)
```

In [12]:
data = bed.reshape(-1,1)

In [13]:
nst_trans_base = QuantileTransformer(n_quantiles=1000, output_distribution='normal', random_state=seed).fit(data)

In [14]:
print("Original Values: ", nst_trans_base.quantiles_[:5], "With Shape:", nst_trans_base.quantiles_.shape)
print("Normalized [0,1] Values: ", nst_trans_base.references_[:5], "With Shape:", nst_trans_base.references_.shape)

Original Values:  [[-1663.05859375]
 [-1397.89825934]
 [-1387.05516698]
 [-1364.34928275]
 [-1333.51492337]] With Shape: (1000, 1)
Normalized [0,1] Values:  [0.       0.001001 0.002002 0.003003 0.004004] With Shape: (1000,)


In [15]:
nst_trans_cp = NormalScoreTransformation(nst_trans_base.quantiles_, nst_trans_base.references_)

In [16]:
sklearn_out = nst_trans_base.transform(data)         # (N, 1) float32
gpu_out     = cp.asnumpy(nst_trans_cp.transform(data))  # same shape/dtype

fwd_diff = np.max(np.abs(sklearn_out.ravel() - gpu_out.ravel()))
print(f"Forward (sklearn vs ours):  max diff = {fwd_diff}")


Forward (sklearn vs ours):  max diff = 0.021762371063232422


In [17]:
x_sk_inv   = nst_trans_base.inverse_transform(sklearn_out)
x_ours_inv = cp.asnumpy(
    nst_trans_cp.inverse_transform(cp.asarray(sklearn_out))
)
inv_diff = np.max(np.abs(x_sk_inv.ravel() - x_ours_inv.ravel()))
print(f"Inverse (sklearn vs ours):  max diff = {inv_diff:}")

Inverse (sklearn vs ours):  max diff = 0.0


Round trip means performing a forward and inverse transformation. A possible reason for the max value of 66 loss between the two methods because of the clipping done by both methods. 

In [18]:

sk_rt   = nst_trans_base.inverse_transform(nst_trans_base.transform(data))
our_rt  = cp.asnumpy(
    nst_trans_cp.inverse_transform(nst_trans_cp.transform(data))
)
print(f"\nRound-trip loss (sklearn): {np.max(np.abs(data - sk_rt))}")
print(f"Round-trip loss (ours):    {np.max(np.abs(data - our_rt))}")


Round-trip loss (sklearn): 163.963623046875
Round-trip loss (ours):    163.963623046875


In [19]:
clip_min = nst_trans_cp._clip_min
clip_max = nst_trans_cp._clip_max
n_clipped = int(np.sum((gpu_out <= clip_min) | (gpu_out >= clip_max)))
print(f"\nValues clamped at z-bounds: {n_clipped} / {gpu_out.size} "
      f"({100*n_clipped/gpu_out.size:.4f}%)")



Values clamped at z-bounds: 81 / 1193314 (0.0068%)


In [20]:
n_trials = 1000
start_time = time.perf_counter()

# Code to measure
for i in range(n_trials):
    sklearn_out = nst_trans_base.transform(data)         # (N, 1) float32
    x_sk_inv   = nst_trans_base.inverse_transform(sklearn_out)


end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"CPU Execution time: {execution_time:.6f} seconds")



start_time = time.perf_counter()

# Code to measure
for i in range(n_trials):
    cond_cupy_mcr = MC_res.get_mass_conservation_residual_fused(cond_bed, bedmap_surf, velx, vely, dhdt, smb, resolution)

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"GPU Execution time: {execution_time:.6f} seconds")



CPU Execution time: 180.387947 seconds
GPU Execution time: 2.418648 seconds


In [21]:
del our_rt
del x_ours_inv
del gpu_out

# Comparing SGS

In [5]:
from MCMC_GPU.SGS_GPU import SGS_MCMC
from gstatsMCMC import MCMC


In [6]:
# [range, sill, shape/smoothness, nugget]
variogram = [9932.545836561178, 1.021964658501033, 1.2259010610301213, 0]

# Matern variogram dict — matches the structure built in chain_sgs_gpu.run() (lines 580-588)
vario = {
    'azimuth'     : 0,
    'nugget'      : variogram[3],          # 0
    'major_range' : variogram[0],          # 9932.5
    'minor_range' : variogram[0],          # same as major
    'sill'        : variogram[1],          # 1.022
    'vtype'       : 'Matern',
    's'           : variogram[2],          # smoothness = 1.226
}

In [7]:
print(cp.cuda.runtime.getDeviceCount())
for i in range(cp.cuda.runtime.getDeviceCount()):
    cp.cuda.Device(i).use()
    free, total = cp.cuda.runtime.memGetInfo()
    props = cp.cuda.runtime.getDeviceProperties(i)
    print(f"Device {i}: {props['name']} | Free: {free/1e9:.2f} GB | Total: {total/1e9:.2f} GB")

1
Device 0: b'NVIDIA L4' | Free: 23.46 GB | Total: 23.66 GB


In [8]:
sgs_cxt = SGS_MCMC(xx, 
                 yy, 
                 vario, 
                 radius = 100e3, 
                 num_points = 32,
                 ktype = 'ok',  # Ordinary krigging
                 seed = None, 
                 batch_size=None, 
                 dtype = cp.float32, 
                 quiet = False, 
                 sigma = 1.5)
sim_mask = cp.full(xx.shape, True) # we can simulation on all cell
bed_cp = cp.array(bed)
assert sim_mask.shape == bed.shape

[SGSContext] batch_size=200000  (stencil ~125663 px)
Default RNG


In [9]:
type(sim_mask), type(bed_cp)

(cupy.ndarray, cupy.ndarray)

In [10]:
block_size = 10
r0, c0 = xx.shape[0] // 2, xx.shape[1] // 2

sim_mask_block = cp.zeros(xx.shape, dtype = bool)
sim_mask_block[r0:r0+block_size, c0:c0+block_size] = True

bed_cp[r0:r0+block_size, c0:c0+block_size] = cp.nan

bed_np = bed.copy()
bed_np[r0:r0+block_size, c0:c0+block_size] = np.nan

sim_mask_np = np.zeros(xx.shape, dtype = bool)
sim_mask_np[r0:r0+block_size, c0:c0+block_size] = True

base_sim = MCMC.sgs(xx, yy, bed_np, vario,
                    radius=100e3, num_points=20, ktype='ok',
                    sim_mask=sim_mask_np, quiet=False, stencil=None,
                    rcond=None, seed=None)
cupy_sim = sgs_cxt.simulate(bed_cp, sim_mask_block)

diff_block = cp.asnumpy(cupy_sim[r0:r0+block_size, c0:c0+block_size]) \
           - base_sim[r0:r0+block_size, c0:c0+block_size]
print("max abs diff in block:", np.max(np.abs(diff_block)))

max abs diff in block: 589.85895


In [ ]:
n_trials = 100
start_time = time.perf_counter()

# Code to measure
for i in range(n_trials):
    base_sim = MCMC.sgs(xx, yy, bed_np, vario,
                    radius=100e3, num_points=20, ktype='ok',
                    sim_mask=sim_mask_np, quiet=False, stencil=None,
                    rcond=None, seed=None)

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"CPU Execution time: {execution_time:.6f} seconds")



start_time = time.perf_counter()

# Code to measure
for i in range(n_trials):

    cupy_sim = sgs_cxt.simulate(bed_cp, sim_mask_block)


end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"GPU Execution time: {execution_time:.6f} seconds")

